# Линейная регрессия: учебная практика

В этом ноутбуке вы разберёте полный базовый пайплайн задачи регрессии:

- постановка задачи;
- загрузка и первичный анализ данных;
- разбиение на train/test;
- обучение линейной регрессии;
- оценка качества через MAE, MSE, RMSE и R²;
- интерпретация коэффициентов модели;
- сравнение с baseline;
- небольшой блок для самостоятельной работы.

Ноутбук рассчитан на первый урок модуля **«Классическое машинное обучение»**.

## 1. Постановка задачи

В задаче регрессии мы хотим предсказывать числовую величину.

Например:

- цену квартиры;
- температуру;
- спрос на товар;
- стоимость машины;
- медицинский показатель.

В этом ноутбуке будем использовать встроенный датасет `diabetes` из `sklearn`.

Целевая переменная — числовой показатель прогрессирования диабета через год после измерений.

Наша цель — обучить линейную модель, которая по признакам пациента предсказывает это значение.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import matplotlib.pyplot as plt

## 2. Загрузка данных

Загрузим датасет и посмотрим, из чего он состоит.

In [ ]:
data = load_diabetes()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("Размер матрицы признаков:", X.shape)
print("Размер целевой переменной:", y.shape)

X.head()

Посмотрим на целевую переменную.

In [ ]:
y.describe()

## 3. Первичный анализ данных

Перед обучением модели полезно понять:

- сколько объектов и признаков есть в датасете;
- есть ли пропуски;
- как распределена целевая переменная;
- какие признаки потенциально связаны с target.

In [ ]:
X.info()

In [ ]:
X.isna().sum()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(y, bins=30)
plt.title("Распределение целевой переменной")
plt.xlabel("target")
plt.ylabel("Количество объектов")
plt.show()

Посмотрим корреляции признаков с целевой переменной.

In [ ]:
df = X.copy()
df["target"] = y

correlations = df.corr(numeric_only=True)["target"].sort_values(ascending=False)
correlations

## 4. Train/test split

Чтобы честно оценить качество модели, нужно разделить данные на обучающую и тестовую части.

- `train` используется для обучения модели;
- `test` используется для финальной оценки качества.

Если оценивать качество только на train, можно получить слишком оптимистичную оценку.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

## 5. Baseline-модель

Перед обучением линейной регрессии полезно построить простой baseline.

Baseline — это простая модель, с которой мы будем сравнивать основную модель.

В задаче регрессии базовый вариант — всегда предсказывать среднее значение target на train.

In [ ]:
baseline_prediction = np.full(shape=len(y_test), fill_value=y_train.mean())

baseline_mae = mean_absolute_error(y_test, baseline_prediction)
baseline_mse = mean_squared_error(y_test, baseline_prediction)
baseline_rmse = np.sqrt(baseline_mse)
baseline_r2 = r2_score(y_test, baseline_prediction)

print("Baseline MAE:", baseline_mae)
print("Baseline MSE:", baseline_mse)
print("Baseline RMSE:", baseline_rmse)
print("Baseline R²:", baseline_r2)

## 6. Обучение линейной регрессии

Линейная регрессия имеет вид:

$$
\hat{y} = w_1x_1 + w_2x_2 + ... + w_dx_d + b
$$

В векторной форме:

$$
\hat{y} = w^Tx + b
$$

Во время обучения модель подбирает веса `w` и свободный коэффициент `b`.

In [ ]:
model = LinearRegression()

model.fit(X_train, y_train)

Сделаем предсказания на train и test.

In [ ]:
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

y_test_pred[:10]

## 7. Метрики качества

Для задачи регрессии часто используют:

### MAE

$$
MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|
$$

Средняя абсолютная ошибка. Хорошо интерпретируется в единицах target.

### MSE

$$
MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2
$$

Средний квадрат ошибки. Сильнее штрафует большие ошибки.

### RMSE

$$
RMSE = \sqrt{MSE}
$$

Корень из MSE. Измеряется в тех же единицах, что и target.

### R²

Показывает, какую долю разброса target объясняет модель.

- `R² = 1` — идеальная модель;
- `R² = 0` — модель не лучше предсказания среднего;
- `R² < 0` — модель хуже baseline по среднему.

In [ ]:
def regression_report(y_true, y_pred, name="model"):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    return pd.Series({
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    }, name=name)

train_report = regression_report(y_train, y_train_pred, "LinearRegression train")
test_report = regression_report(y_test, y_test_pred, "LinearRegression test")
baseline_report = pd.Series({
    "MAE": baseline_mae,
    "MSE": baseline_mse,
    "RMSE": baseline_rmse,
    "R2": baseline_r2
}, name="Baseline test")

reports = pd.DataFrame([baseline_report, train_report, test_report])
reports

## 8. Сравнение фактических и предсказанных значений

Если модель работает хорошо, точки на графике должны лежать близко к диагонали.

Диагональ означает идеальное совпадение:

$$
y = \hat{y}
$$

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_test_pred, alpha=0.7)

min_value = min(y_test.min(), y_test_pred.min())
max_value = max(y_test.max(), y_test_pred.max())

plt.plot([min_value, max_value], [min_value, max_value])
plt.title("Фактические значения vs предсказания")
plt.xlabel("Фактическое значение")
plt.ylabel("Предсказание модели")
plt.show()

## 9. Анализ ошибок

Посмотрим на распределение ошибок модели.

Ошибка для объекта:

$$
error_i = y_i - \hat{y}_i
$$

Если ошибки распределены примерно вокруг нуля, это хороший знак.

In [ ]:
errors = y_test - y_test_pred

plt.figure(figsize=(8, 5))
plt.hist(errors, bins=30)
plt.title("Распределение ошибок")
plt.xlabel("Ошибка: y_true - y_pred")
plt.ylabel("Количество объектов")
plt.show()

errors.describe()

## 10. Интерпретация коэффициентов

У линейной регрессии можно посмотреть коэффициенты.

Коэффициент показывает, как меняется предсказание при увеличении признака на одну единицу при прочих равных.

Важно: в этом датасете признаки уже стандартизованы, поэтому коэффициенты можно сравнивать между собой осторожно, но удобнее, чем для признаков в разных масштабах.

In [ ]:
coefficients = pd.Series(model.coef_, index=X.columns).sort_values(key=abs, ascending=False)

coefficients

In [ ]:
plt.figure(figsize=(10, 5))
coefficients.sort_values().plot(kind="barh")
plt.title("Коэффициенты линейной регрессии")
plt.xlabel("Значение коэффициента")
plt.ylabel("Признак")
plt.show()

## 11. Выводы

Ответьте на вопросы:

1. Модель лучше baseline или нет?
2. Какая метрика наиболее понятна для интерпретации?
3. Насколько сильно качество на train отличается от качества на test?
4. Есть ли признаки сильного переобучения?
5. Какие признаки имеют наибольшие по модулю коэффициенты?

In [ ]:
# Напишите свои выводы здесь в виде комментариев

# Мои выводы:
# 1.
# 2.
# 3.
# 4.
# 5.

# Самостоятельная работа

Выполните задания ниже.

## Задание 1

Обучите линейную регрессию только на одном самом коррелирующем признаке.

Сравните качество с моделью, которая использует все признаки.

## Задание 2

Постройте график зависимости target от этого признака и добавьте предсказания модели.

## Задание 3

Попробуйте другое значение `test_size`, например `0.3`.

Поменялось ли качество?

## Задание 4

Сделайте финальный вывод:

- какая модель лучше;
- насколько она лучше baseline;
- какие ошибки у модели самые заметные.

In [ ]:
# Задание 1

top_feature = correlations.drop("target").abs().sort_values(ascending=False).index[0]
top_feature

In [ ]:
# Обучите модель на одном признаке

X_one_train = X_train[[top_feature]]
X_one_test = X_test[[top_feature]]

one_feature_model = LinearRegression()
one_feature_model.fit(X_one_train, y_train)

y_one_pred = one_feature_model.predict(X_one_test)

one_feature_report = regression_report(y_test, y_one_pred, f"One feature: {top_feature}")
one_feature_report

In [ ]:
# Сравните результаты

comparison = pd.DataFrame([
    baseline_report,
    test_report,
    one_feature_report
])

comparison

In [ ]:
# Задание 2

plt.figure(figsize=(8, 5))
plt.scatter(X_one_test[top_feature], y_test, alpha=0.7, label="Истинные значения")
plt.scatter(X_one_test[top_feature], y_one_pred, alpha=0.7, label="Предсказания")
plt.title(f"Линейная регрессия по признаку {top_feature}")
plt.xlabel(top_feature)
plt.ylabel("target")
plt.legend()
plt.show()

## Критерии оценки

Всего: **45 баллов**.

- корректная загрузка и подготовка данных — 10 баллов;
- обучение линейной регрессии — 10 баллов;
- расчёт MAE, MSE, RMSE и R² — 10 баллов;
- сравнение с baseline — 5 баллов;
- анализ коэффициентов и ошибок — 5 баллов;
- выводы по результатам — 5 баллов.

## Что важно запомнить

1. Линейная регрессия — базовая модель для задачи регрессии.
2. Перед обучением модель нужно сравнивать с baseline.
3. Качество нужно оценивать на test, а не только на train.
4. MAE проще интерпретировать, MSE сильнее штрафует большие ошибки.
5. Коэффициенты линейной модели можно использовать для интерпретации, но нужно помнить про масштаб признаков.